dummy code


In [ ]:
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
import matplotlib.pyplot as plt

# Define the function to create and train the model
def train_model(filters, kernel_size, dense_units, dropout_rate, learning_rate, epochs=10):
    model = Sequential([
        Conv2D(filters=filters, kernel_size=kernel_size, activation='relu', input_shape=(28,28,1)),
        MaxPooling2D(pool_size=(2,2)),
        Conv2D(filters=filters*2, kernel_size=kernel_size, activation='relu'),
        MaxPooling2D(pool_size=(2,2)),
        Flatten(),
        Dense(dense_units, activation='relu'),
        Dropout(dropout_rate),
        Dense(dense_units // 2, activation='relu'),
        Dense(10, activation='softmax')  # 10 output classes
    ])
    
    model.compile(optimizer=Adam(learning_rate=learning_rate), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    
    # Early stopping to prevent overfitting
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    
    history = model.fit(train_images, y_train, epochs=epochs, 
                        validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    return model, history

# Define combinations of parameters to try
combinations = [
    {'filters': 32, 'kernel_size': (3,3), 'dense_units': 128, 'dropout_rate': 0.5, 'learning_rate': 0.0005},
    {'filters': 64, 'kernel_size': (3,3), 'dense_units': 256, 'dropout_rate': 0.3, 'learning_rate': 0.001},
]

results = []
for i, combo in enumerate(combinations):
    model, history = train_model(**combo, epochs=20)
    test_loss, test_accuracy = model.evaluate(test_images, y_test, verbose=0)
    
    results.append({
        'config': combo,
        'model': model,
        'history': history,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy
    })
    
    # Plot accuracy and loss for each configuration
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f"Configuration {i+1} - Accuracy")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f"Configuration {i+1} - Loss")
    plt.legend()
    
    plt.show()
    
    
    


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Choose 10 random indices
random_indices = np.random.choice(len(test_images), size=10, replace=False)

# Plot predictions
for i, idx in enumerate(random_indices):
    image = test_images[idx]
    true_label = y_test[idx]
    prediction = model.predict(image.reshape(1, 28, 28, 1))
    predicted_label = np.argmax(prediction)
    
    plt.figure(figsize=(3, 3))
    plt.imshow(image.reshape(28, 28), cmap='gray')
    plt.title(f"True Label: {true_label} - Predicted: {predicted_label}")
    plt.xlabel(f"Probabilities: {prediction[0]}")
    plt.show()
